# 73 — M8 Matrix & PIT Audit

Une las familias avanzadas en TRAIN/VALIDATION y audita leakage temporal, aislamiento de holdouts e integridad del protocolo M7.


In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.advanced import load_advanced_spec
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()
M8_SPEC = load_advanced_spec(paths.configs / "advanced.toml")
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MANIFEST_DIR = paths.root / "artifacts" / "manifests"
print("M8 schema:", M8_SPEC.schema_version)
from argos.config.production import load_production_spec
from argos.config.prospective import load_prospective_spec
M6_SPEC = load_production_spec(paths.configs / "production.toml")
M7_SPEC = load_prospective_spec(paths.configs / "prospective.toml")


In [ ]:
from argos.advanced.m8 import build_m8_development_matrix, build_m8_pit_audit

matrix = build_m8_development_matrix(
    root=paths.root,
    reports_root=paths.reports,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    advanced_spec=M8_SPEC,
)
assert matrix.ready_for_pit_audit
pit = build_m8_pit_audit(
    root=paths.root,
    reports_root=paths.reports,
    manifest_dir=MANIFEST_DIR,
    modeling_spec=MODEL_SPEC,
    advanced_spec=M8_SPEC,
    prospective_spec=M7_SPEC,
    production_spec=M6_SPEC,
    docs_root=paths.docs,
)
pit.validation


In [ ]:
blocking = pit.validation.loc[pit.validation["blocking"]]
print("73 ✅ PASS" if pit.ready_for_ablation else "73 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
assert pit.ready_for_ablation
